# **Escuela en Computación Cuántica para Informáticos I** (2026)- **UNRN**
# **Modulo I:** Introducción a la computación cuántica y aplicaciones prácticas
**Docente:** Dr. Andrés Alejandro Reynoso - **Ayudante:** Enrique Nicanor Mariotti

## **Día II:** Instalacion de Qiskit

Si tenemos instalado qiskit, ejecutamos la celda de abajo. Si es la primera vez que lo instalamos, no es necesario.

In [ ]:
# 1. Desinstalamos el paquete anterior
#!pip uninstall -y qiskit-transpiler-service

Instalar qiskit en su versión 2.4.2

In [ ]:
# 2. Instalamos Qiskit 2.4.2 y las librerías de IBM Runtime actualizadas
!pip install qiskit==2.4.2 qiskit-ibm-runtime samplomatic

Luego instalar estas otras librerías que son necesarias.

In [ ]:
%pip install qiskit_aer
%pip install qiskit_ibm_runtime
%pip install matplotlib
%pip install pylatexenc

**NOTA:** Testeamos la instalación reiniciando el entorno desde "Entorno de ejecución" -> "Reiniciar la sesión"

Probamos que se haya instalado correctamente e importamos las librerías.

In [ ]:
import qiskit
print("Qiskit versión:", qiskit.__version__)

import qiskit_ibm_runtime
print("¡Todo listo y sin conflictos!")

Probamos crear un circuito simple para probar que este funcionando todo.


In [ ]:
from qiskit import QuantumCircuit

# Crear un nuevo circuito con 1 qubit
qc = QuantumCircuit(1)

# Aplicar una compuerta X (NOT) al qubit 0
qc.x(0)

# Retorna un dibujo del circuito con MatPlotLib ("mpl").
# Esta es la ultima linea de codigo para que el grafico aparezca en la celda.
qc.draw("mpl")

**NOTA:** Cada vez que uses qiskit en Colab (en una nueva sesión) vamos a tener que instalarlo, ya que no es una librería "preinstalada". Ejecutá las celdas 2 y 3, reiniciá el entorno de ejecución y después verificá ejecutanto la celda 4. Ahora podemos seguir adelante.

In [ ]:
# Configuración general: import y simulador
import random
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

from qiskit import QuantumCircuit, transpile
from qiskit import ClassicalRegister, QuantumRegister

from qiskit_aer import AerSimulator
from qiskit.circuit import Parameter
from qiskit.primitives import StatevectorSampler
from qiskit.quantum_info import Operator, Statevector
from qiskit.visualization import plot_histogram

sim = AerSimulator()

def run_circuit(qc, shots=1024):
    """Transpila y ejecuta un circuito en el simulador AerSimulator, devuelve counts."""
    tqc = transpile(qc, sim)
    result = sim.run(tqc, shots=shots).result()
    return result.get_counts()

## **Dia II:** Algoritmos de Deutsch y Deutsch-Jozsa

### **Algoritmo de Deutsch:** las 4 funciones f:{0,1}→{0,1}

Evaluamos sobre alguna funciones construidas por nosotros mismos:

- **Funcion igual a 0**:  (constante) hay independencia entre x e y.
- **Funcion igual a  1**: (constante) hay independencia entre x e y, pero y se invierte.
- **Funcion Identidad**: (balanceada): es equivalente a CNOT(x, y).
- **Funcion Inversión**: (balanceada): es equivalente a X(x) + CNOT(x, y) + X(x).

Al medir el qubit x: `0` indica función constante, `1` indica función balanceada.

In [ ]:
def deutsch(oracle_fn):
    qc = QuantumCircuit(2, 1)
    oracle = QuantumCircuit(2, name="Oráculo")
    qc.x(1)                                    # y = |1>
    qc.h(0); qc.h(1)                           # Superposición en x e y
    oracle_fn(oracle)                          # Aplica el oráculo F correspondiente
    qc.append(oracle.to_instruction(), [0, 1]) # Encapsulamos el grafico
    qc.h(0)                                    # H final sobre x
    qc.measure(0, 0)
    display(qc.draw(output='mpl'))
    return run_circuit(qc, shots=1024)

def oracle_const0(qc):
    pass                           # Variables independientes:f(x) = 0, es decir, y ⊕ f(x) = y

def oracle_const1(qc):
    qc.x(1)                        # Invierte y: f(x) = 1, es decir, y ⊕ f(x) = ¬y

def oracle_identity(qc):
    qc.cx(0, 1)                    # Funcion balanceada: f(x) = x, es decir, y ⊕ f(x) = y ⊕ x

def oracle_inversion(qc):
    qc.x(0); qc.cx(0, 1); qc.x(0)  # Funcion balanceada: f(x) = ¬x, es decir,  y ⊕ f(x) = y ⊕ ¬x

print("Deutsch - constante 0  :", deutsch(oracle_const0),    " (esperado: '0')")
print("Deutsch - constante 1  :", deutsch(oracle_const1),    " (esperado: '0')")
print("Deutsch - identidad x  :", deutsch(oracle_identity),  " (esperado: '1')")
print("Deutsch - inversión ¬x :", deutsch(oracle_inversion), " (esperado: '1')")


### **Algoritmo de Deutsch-Jozsa:** n=3 qubits

Generalización a `f: {0,1}^n -> {0,1}`. Implementamos la función constante-0 y la función balanceada `f(x0,x1,x2) = x0 ⊕ x1 ⊕ x2`. Si el resultado es `000`, la función es constante; cualquier otro resultado indica balanceada.

In [ ]:
def deutsch_jozsa(oracle_fn, n = 3):
    qc = QuantumCircuit(n + 1, n)
    oracle = QuantumCircuit(n + 1, name="Oráculo")
    qc.x(n)                 # Ultimo qubit = |1>
    qc.h(range(n + 1))      # H en todos los qubits
    oracle_fn(oracle, n)    # Aplica el oráculo F correspondiente
    qc.append(oracle.to_instruction(), range(n+1)) # Encapsulamos el grafico
    qc.h(range(n))          # H final en los qubits x
    qc.measure(range(n), range(n))
    display(qc.draw(output='mpl'))
    return run_circuit(qc, shots=1024)

def dj_oracle_const0(qc, n):
    print("\nImplentacion del Oráculo:")
    display(qc.draw(output='mpl'))
    pass                    # Variables independientes:f(x) = 0, es decir, y ⊕ f(x) = y

def dj_oracle_balanced_xor(qc, n):
    for i in range(n):
        qc.cx(i, n)         # Funcion balanceada: f(x0,x1,x2) = x0 ⊕ x1 ⊕ x2, es decir, y ⊕ f(x) = y  ⊕ x0 ⊕ x1 ⊕ x2
    print("\nImplentacion del Oráculo:")
    display(qc.draw(output='mpl'))


print("Deutsch-Jozsa - constante f(x) = 0:", deutsch_jozsa(dj_oracle_const0),
      " (esperado: '000' -> constante)")
print("Deutsch-Jozsa - balanceada f(x) = x0 ⊕ x1 ⊕ x2:", deutsch_jozsa(dj_oracle_balanced_xor),
      " (esperado: distinto de '000' -> balanceada)")

## **Dia II:** Algoritmo de Simon

### **Algoritmo de Simon:**  Ejemplo de 2 qubits con s = "10".

Circuito de 2 qubits que emula el oráculo de Simon con cadena secreta `s=10`. El
resultado siempre es `00` o `10` (nunca `01` ni `11`) en notación Qiskit. Usando la
salida útil `z = z0z1 = 01`, se resuelve manualmente la ecuación
`z . s = z0*s0 + z1*s1 = 0*s0 + 1*s1 = s1 = 0 (mod 2)` para deducir `s1 = 0`, y como
`s` no puede ser todo ceros, `s0 = 1`, concluyendo que `s = s0s1 = 10`.

In [ ]:
# Oráculo simplificado de 2 qubits para s = 10:
# Aquí lo modelamos con un registro y explícito de 2 qubits (x0,x1,y0,y1)

qc_simon2 = QuantumCircuit(4, 2)
oraculo = QuantumCircuit(4, name="Operador Oráculo")

qc_simon2.h(0); qc_simon2.h(1)  # superposición en x0, x1

# Oráculo para s = "10"
oraculo.cx(0, 2)  # y0 = x0
oraculo.cx(1, 3)  # y1 = x1
oraculo.cx(0, 2)
qc_simon2.append(oraculo.to_instruction(), [0, 1, 2, 3]) # Encapsulamos el grafico

qc_simon2.h(0); qc_simon2.h(1)
qc_simon2.measure([0, 1], [0, 1])

print("Circuito de Simon:")
display(qc_simon2.draw(output='mpl'))
print("Operador Oráculo:")
display(oraculo.draw(output='mpl'))
print("Resultado (solo '00' o '10', nunca '01' o'11'):", run_circuit(qc_simon2, shots=1024))
print("Recordar que la notacion de qiskit es q1q0.")

### **Algoritmo de Simon:**  Construcción general del oráculo.

Algoritmo genérico para construir el oráculo de Simon dada una cadena secreta `s`:

1. Aplicar CNOT desde cada qubit del registro `x` al qubit correspondiente del registro `y`.
2. Encontrar el índice `k` más chico tal que `s[k] = 1`. Aplicar CNOT desde `x[k]` hacia cada `y[i]` tal que `s[i] = 1`.

Implementamos como una función reutilizable y lo probamos con `s = 110`.

In [ ]:
def simon_oracle(qc, s, n):
    """Construye el oráculo de Simon para la cadena secreta s (string de longitud n).
    Los qubits 0..n-1 son el registro x, n..2n-1 son el registro y.
    """
    # Paso 1: Copiar x a y
    for i in range(n):
        qc.cx(i, n + i)
    # Paso 2: Encontrar el índice k más chico con s[k]='1'. Aplicar CNOT desde `x[k]` hacia cada `y[i]` tal que `s[i] = 1`.
    k = s.index('1')
    for i in range(n):
        if s[i] == '1':
            qc.cx(k, n + i)

def simon_full_circuit(s):
    n = len(s)
    qc = QuantumCircuit(2 * n, n)
    oraculo = QuantumCircuit(2 * n, name="Operador Oráculo")
    qc.h(range(n))
    simon_oracle(oraculo, s, n)
    qc.append(oraculo.to_instruction(), range(2 * n))
    qc.h(range(n))
    qc.measure(range(n), range(n))
    return qc, oraculo

qc_simon_s110, oraculo = simon_full_circuit("110")

print("Circuito de Simon:")
display(qc_simon_s110.draw(output='mpl'))
print("Operador Oráculo:")
display(oraculo.draw(output='mpl'))

counts = run_circuit(qc_simon_s110, shots=2048)
print("\nResultados:", counts)

# Verificación: para cada z medido, comprobar el producto punto con s.
# Nota sobre el orden de bits: Qiskit reporta el bitstring de counts con el bit del
# qubit de índice más alto a la izquierda (c[n-1]...c[0]), así que antes de leerlo
# como z0 z1 ... zn-1 hay que invertir el string.
s = "110"
print(f"\nVerificación z . s = 0 (mod 2), con s = {s}:")
for z in counts:
    z_normal_order = z[::-1]  # ahora z_normal_order[i] = valor medido del qubit i = z_i
    dot = sum(int(z_normal_order[i]) * int(s[i]) for i in range(len(s))) % 2
    print(f"  bitstring Qiskit={z}  (z0z1z2={z_normal_order})  ->  z . s mod 2 = {dot}  (debe ser 0)")

## **Dia II:** Algoritmo de Búsqueda de Grover

### **Algoritmo de Grover:** Búsqueda de 2 bits, elemento objetivo = 11

Buscamos la llave dada por la expresion booleana `q0 ^ q1`, es decir el objetivo es `11`.

#### **Operador Oráculo**

El operador oráculo es:

$$U_{\omega} = I - 2|\omega \rangle\langle \omega|$$

La compuerta **CZ** actúa sobre 2 qubits y es simétrica entre control y target: invierte el signo del estado únicamente cuando **ambos** qubits están en $|1\rangle$, y deja todos los demás estados sin cambios.

$$\text{CZ}|q_0 q_1\rangle = (-1)^{q_0 \cdot q_1}\,|q_0 q_1\rangle$$

Explícitamente:

$$\text{CZ}|00\rangle = |00\rangle, \quad \text{CZ}|01\rangle = |01\rangle, \quad \text{CZ}|10\rangle = |10\rangle, \quad \text{CZ}|11\rangle = -|11\rangle$$

En forma matricial (base $00,01,10,11$):

$$\text{CZ} = \begin{pmatrix} 1&0&0&0 \\ 0&1&0&0 \\ 0&0&1&0 \\ 0&0&0&-1 \end{pmatrix}$$

#### **Operador Difusión**

El operador de difusión se define formalmente como una **reflexión respecto al estado de superposición uniforme** $|s\rangle$:

$$U_s = 2|s\rangle\langle s| - I$$

donde $|s\rangle = H^{\otimes n}|0\rangle^{\otimes n}$ es el estado que se obtiene aplicando Hadamard a todos los qubits.

**Paso 1: Sustituir $|s\rangle$ y sacar factor común $H^{\otimes n}$:**

Como $H^{\otimes n}$ es su propia inversa ($H^{\otimes n} H^{\otimes n} = I$), podemos reescribir:

$$U_s = 2H^{\otimes n}|0\rangle\langle 0|H^{\otimes n} - I = H^{\otimes n}\Big(2|0\rangle\langle 0| - I\Big)H^{\otimes n}$$

Esto ya explica la estructura general **H ... H** del circuito: todo el trabajo ocurre en el operador central

$$D_0 = 2|0\rangle\langle 0| - I$$

**Paso 2: $D_0$:**

$D_0$ es diagonal en la base computacional:

$$D_0|0\rangle^{\otimes n} = +|0\rangle^{\otimes n}, \qquad D_0|x\rangle = -|x\rangle \ \ \text{para todo } x \neq 0$$

Es decir, deja intacto a $|0\ldots0\rangle$ e invierte la fase de todos los demás estados.

**Paso 3: Implementar $D_0$:**

La compuerta $X^{\otimes n}$ intercambia $|0\rangle^{\otimes n} \leftrightarrow |1\rangle^{\otimes n}$, así que:

$$X^{\otimes n} D_0 X^{\otimes n} = 2|1\rangle^{\otimes n}\langle 1|^{\otimes n} - I$$

que es (salvo una fase global no observable) exactamente lo que hace una compuerta **Controlled-Z** multi-controlada: deja todos los estados sin cambios excepto $|11\ldots1\rangle$, al cual le invierte el signo.

$$\text{CZ}|11\rangle = -|11\rangle, \qquad \text{CZ}|x\rangle = |x\rangle \ \ \text{si } x \neq 11$$

Por lo tanto:

$$D_0 \ \propto\ X^{\otimes n}\,\text{CZ}\,X^{\otimes n}$$

Finalmente:

$$U_s \;=\; H^{\otimes n}\,X^{\otimes n}\,\text{CZ}\,X^{\otimes n}\,H^{\otimes n}$$

In [ ]:
qc_grover2 = QuantumCircuit(2, 2)
oraculo = QuantumCircuit(2, name="Operador Oráculo")
difusion = QuantumCircuit(2, name="Operador Difusión")

# Transformada de Hadamard inicial
qc_grover2.h([0, 1])

# Oráculo: invierte la fase de |11>
oraculo.cz(0, 1)
qc_grover2.append(oraculo.to_instruction(), [0, 1]) # Encapsulamos el grafico

# Amplificación de amplitud (difusión)
difusion.h([0, 1])
difusion.x([0, 1])
difusion.cz(0, 1)
difusion.x([0, 1])
difusion.h([0, 1])
qc_grover2.append(difusion.to_instruction(), [0, 1]) # Encapsulamos el grafico

qc_grover2.measure([0, 1], [0, 1])

print("Circuito de Groover:")
display(qc_grover2.draw(output='mpl'))
print("Operador Oráculo:")
display(oraculo.draw(output='mpl'))
print("Operador Difusión:")
display(difusion.draw(output='mpl'))
counts = run_circuit(qc_grover2, shots=1024)
print("\nResultado (debería concentrarse en '11'):", counts)
plot_histogram(counts, title="Grover 2-qubit, objetivo=11")


In [ ]:
# Verificación de la matriz del oráculo y de la amplificación (difusión)
qc_oracle_only = QuantumCircuit(2)
qc_oracle_only.cz(0, 1)
print("Matriz del oráculo (Controlled-Z):")
print(Operator(qc_oracle_only).data)

qc_diffusion_only = QuantumCircuit(2)
qc_diffusion_only.h([0, 1])
qc_diffusion_only.x([0, 1])
qc_diffusion_only.cz(0, 1)
qc_diffusion_only.x([0, 1])
qc_diffusion_only.h([0, 1])
print("\nMatriz de difusión:")
print(Operator(qc_diffusion_only).data)


### **Algoritmo de Grover:** búsqueda de 3 bits, elemento objetivo = 111

Buscamos la llave dada por la expresion booleana `q0 ^ q1 ^ q2`, es decir el objetivo es `111`. Ya que Qiskit no tiene una puerta `CCZ` nativa, se emula con `H - CCX - H`. Es decir, `H·X·H` sobre el qubit objetivo, rodeando a la puerta Toffoli, es equivalente a una puerta Z controlada dos veces. Se muestra que la amplitud en `111` domina sobre el resto.

In [ ]:
qc_grover3 = QuantumCircuit(3, 3)
oraculo = QuantumCircuit(3, name="Operador Oráculo")
difusion = QuantumCircuit(3, name="Operador Difusión")

# H inicial
qc_grover3.h([0, 1, 2])

# Oráculo: CCZ emulado con H-CCX-H sobre el qubit 2 (target)
oraculo.h(2)
oraculo.ccx(0, 1, 2)
oraculo.h(2)
qc_grover3.append(oraculo.to_instruction(), [0, 1, 2]) # Encapsulamos el grafico


# Amplificación de amplitud (difusión)
difusion.h([0, 1, 2])
difusion.x([0, 1, 2])
difusion.h(2)
difusion.ccx(0, 1, 2)
difusion.h(2)
difusion.x([0, 1, 2])
difusion.h([0, 1, 2])
qc_grover3.append(difusion.to_instruction(), [0, 1, 2]) # Encapsulamos el grafico

qc_grover3.measure([0, 1, 2], [0, 1, 2])

display(qc_grover3.draw(output='mpl'))
counts = run_circuit(qc_grover3, shots=2048)
print("\nResultado (la mayor amplitud debería estar en '111'):", counts)
plot_histogram(counts, title="Grover 3-qubit, objetivo=111")

In [ ]:
# Verificación de la matriz del oráculo y de la amplificación (difusión)
qc_oracle_only = QuantumCircuit(3)
qc_oracle_only.h(2)
qc_oracle_only.ccx(0, 1, 2)
qc_oracle_only.h(2)
print("Matriz del oráculo (Controlled-Z):")
print(Operator(qc_oracle_only).data.real)

qc_diffusion_only = QuantumCircuit(3)
qc_diffusion_only.h([0, 1, 2])
qc_diffusion_only.x([0, 1, 2])
qc_diffusion_only.h(2)
qc_diffusion_only.ccx(0, 1, 2)
qc_diffusion_only.h(2)
qc_diffusion_only.x([0, 1, 2])
qc_diffusion_only.h([0, 1, 2])
print("\nMatriz de difusión:")
print(Operator(qc_diffusion_only).data.real)

### **Algoritmo de Grover:** Repetición del oráculo + amplificación usando `repeat()`

Una mejora de performance posible sería usar el método `repeat()` de Qiskit para repetir el operador oráculo y el operador difusión sin que el *transpiler* duplique manualmente el circuito, lo que evitaría un crecimiento exponencial del número de puertas.

En este experimento construimos el oráculo y la amplificación como sub-circuitos y los repetimos distinta cantidad de veces para observar el fenómeno de "sobrecocción" (overcooking). Es decir, repetir demasiadas veces reduce la probabilidad del estado deseado en vez de aumentarla.

In [ ]:
from qiskit.circuit import Gate

def build_oracle_gate():
    qc = QuantumCircuit(3, name="Oráculo")
    qc.h(2)
    qc.ccx(0, 1, 2)
    qc.h(2)
    return qc.to_gate()

def build_diffusion_gate():
    qc = QuantumCircuit(3, name="Difusión")
    qc.h([0, 1, 2])
    qc.x([0, 1, 2])
    qc.h(2)
    qc.ccx(0, 1, 2)
    qc.h(2)
    qc.x([0, 1, 2])
    qc.h([0, 1, 2])
    return qc.to_gate()

def grover_repeated(num_iterations):
    oracle_gate = build_oracle_gate()
    diffusion_gate = build_diffusion_gate()

    qc = QuantumCircuit(3, 3)
    qc.h([0, 1, 2])

    # Combinar oráculo + difusión en una sola puerta compuesta y repetirla con .repeat()
    combined = QuantumCircuit(3, name="Grover_step")
    combined.append(oracle_gate, [0, 1, 2])
    combined.append(diffusion_gate, [0, 1, 2])
    combined_gate = combined.to_gate()

    repeated_gate = combined_gate.repeat(num_iterations)
    qc.append(repeated_gate, [0, 1, 2])

    qc.measure([0, 1, 2], [0, 1, 2])
    return qc

for iterations in [1, 2, 3, 4, 5, 6, 7, 8]:
    qc = grover_repeated(iterations)
    counts = run_circuit(qc, shots=2048)
    prob_111 = counts.get('111', 0) / sum(counts.values())
    print(f"Iteraciones={iterations}: P(111) = {prob_111:.3f}   counts={counts}")

print("\nNotar que para 3 qubits, el número óptimo de iteraciones de Grover es 2,")
print("mas iteraciones 'sobrecocinan' la búsqueda y bajan la probabilidad de 111.")


 ## **Referencias**

Notebook basada en el paper **"Understanding Quantum Parallelism through Programming"** (Sang & Yu, CSCI 2023), implementando en **Qiskit 2.4.2** los circuitos descritos en la Sección III, Labs 1 a 7. Se han agregado experimentos adicionales para incorporar otros temas.